# Day 35: Phase 2 retrospective and release

**Phase 2: EDA & Feature Engineering**

## Objective
Phase 2 retrospective and release

## Task
Audit Phase 2 deliverables. Ensure all notebooks run clean. Cleanup code. Commit and tag v0.2-eda-features.

**Deliverable:** Release v0.2 + Phase 2 summary in journal.

---

### Instructions:
*Treat this as your course workbook. Write your code, notes, or execution steps below.*

Remember: 
- Document your decisions.
- Use clear variable names.
- If today's task involves creating a script in `src/`, a dashboard in `apps/`, or documentation in `docs/`, you can use this notebook to test your logic or simply write your reflections and proofs of execution (e.g. running terminal commands with `!`).



## 1. Phase 2 Deliverable Inventory & Artifact Audit (Days 16-34)

**Objective:** Programmatically inventory all Phase 2 notebooks (`day016` through `day035`), modular Python packages in `src/`, generated `.parquet` feature stores in `data/`, and publication assets in `docs/` before running hygiene and execution checks.

**Engineering Rationale:** Releasing `v0.2-eda-features` requires deterministic verification that every planned deliverable exists on disk, follows naming conventions, and contains no missing artifacts prior to Phase 3 modeling.

In [ ]:
from pathlib import Path

cwd = Path.cwd()
REPO_ROOT = cwd if (cwd / "data").exists() else cwd.parent.parent

nb_dir = REPO_ROOT / "notebooks" / "phase2_eda_features"
notebooks = sorted(nb_dir.glob("day*.ipynb"))
print(f"Phase 2 notebooks found: {len(notebooks)} (expected 20: day016-day035)")
for nb in notebooks:
    print(f" - {nb.name} ({nb.stat().st_size / 1024:.1f} KB)")

print("\n--- Modular Source Files (src/) ---")
for py in sorted((REPO_ROOT / "src").rglob("*.py")):
    if "__pycache__" not in str(py) and py.name != "__init__.py":
        print(f" - {py.relative_to(REPO_ROOT)}")

print("\n--- Feature Stores & Processed Parquet Files ---")
for pq in sorted((REPO_ROOT / "data").rglob("*.parquet")):
    if "raw" not in pq.parts:
        print(f" - {pq.relative_to(REPO_ROOT)} ({pq.stat().st_size / (1024*1024):.2f} MB)")

print("\n--- Web & Tracker Deliverables (docs/) ---")
for rel in ["docs/index.html", "docs/footballscout.html", "docs/phase2-journal.html", "docs/_data/progress.yml"]:
    p = REPO_ROOT / rel
    print(f" - {rel}: exists={p.exists()}")


Phase 2 notebooks found: 20 (expected 20: day016-day035)
 - day016_shot_analysis_eda.ipynb (696.8 KB)
 - day017_xg_features_part_1.ipynb (1104.9 KB)
 - day018_xg_geometrical_features.ipynb (306.7 KB)
 - day019_passing_networks.ipynb (251.9 KB)
 - day020_nfl_offensive_eda.ipynb (566.4 KB)
 - day021_nfl_player_features.ipynb (704.9 KB)
 - day022_rolling_features.ipynb (419.8 KB)
 - day023_per_90_and_radar_charts.ipynb (1331.5 KB)
 - day024_pressing_analysis.ipynb (497.1 KB)
 - day025_nfl_situational_analysis.ipynb (196.8 KB)
 - day026_player_heatmaps_kde.ipynb (2314.0 KB)
 - day027_player_clustering.ipynb (747.8 KB)
 - day028_player_similarity_engine.ipynb (227.9 KB)
 - day029_project_1_readme.ipynb (264.2 KB)
 - day030_nfl_draft_combine_eda.ipynb (436.2 KB)
 - day031_draft_features.ipynb (198.2 KB)
 - day032_possession_analysis.ipynb (31.5 KB)
 - day033_formation_analysis.ipynb (143.2 KB)
 - day034_journal_post_phase_2.ipynb (19.6 KB)
 - day035_phase_2_retrospective.ipynb (7.0 KB)

--- 

### Key Takeaways: Deliverable Inventory

- **100% artifact completeness:** All 20 Phase 2 notebooks (`day016` to `day035`), 20 modular Python files under `src/`, 18 processed/feature `.parquet` files, and all web/tracker deliverables in `docs/` are present on disk.
- **Repository weight check:** Processed feature stores are compact (most under 1 MB; `fact_events.parquet` at 9.48 MB), so we must verify `.gitignore` coverage for raw data and large caches before tagging.

## 2. Schema Standardization & Editorial Hygiene Audit (Days 16-35)

**Objective:** Scan every Phase 2 notebook (`day016` to `day035`) for reproducibility and portfolio-grade standards:
1. Hardcoded local paths (`C:\Users\` or `c:/Users/`).
2. Conversational artifacts (`as requested`, `you asked`, `step 1 from chat`, `porfa`, `chat`) or placeholder cells (`# Start working here...`).
3. Structural schema compliance: every code cell should be preceded by an explanatory Markdown cell, and the notebook should close with a concluding Markdown summary.

**Engineering Rationale:** Portfolio notebooks must read as autonomous, production-grade technical documents rather than interactive chat transcripts.

In [ ]:
import json
import re
import pandas as pd

CONV_PATTERN = re.compile(r"\b(as requested|you asked|from chat|porfa|jeje)\b", re.IGNORECASE)
PATH_PATTERN = re.compile(r"[cC]:[/\\\\]Users[/\\\\]")

audit_rows = []
for nb_path in notebooks:
    if nb_path.name == "day035_phase_2_retrospective.ipynb":
        continue
    nb = json.loads(nb_path.read_text(encoding="utf-8"))
    cells = nb.get("cells", [])
    n_code = sum(1 for c in cells if c["cell_type"] == "code")
    n_md = sum(1 for c in cells if c["cell_type"] == "markdown")

    abs_paths = 0
    conv_hits = 0
    empty_code = 0
    unpaired_code = 0

    for idx, cell in enumerate(cells):
        src = "".join(cell.get("source", []))
        if PATH_PATTERN.search(src):
            abs_paths += 1
        if CONV_PATTERN.search(src):
            conv_hits += 1
        if cell["cell_type"] == "code":
            stripped = src.strip()
            if not stripped or stripped == "# Start working here...":
                empty_code += 1
            if idx == 0 or cells[idx - 1]["cell_type"] != "markdown":
                unpaired_code += 1

    ends_with_md = bool(cells) and cells[-1]["cell_type"] == "markdown"
    audit_rows.append({
        "notebook": nb_path.name,
        "code_cells": n_code,
        "md_cells": n_md,
        "abs_paths": abs_paths,
        "conv_hits": conv_hits,
        "empty_code": empty_code,
        "unpaired_code": unpaired_code,
        "ends_with_md": ends_with_md,
    })

audit_df = pd.DataFrame(audit_rows)
print(audit_df.to_string(index=False))

issues = audit_df[(audit_df["abs_paths"] > 0) | (audit_df["conv_hits"] > 0) | (audit_df["empty_code"] > 0) | (audit_df["unpaired_code"] > 0) | (~audit_df["ends_with_md"])]
print(f"\nNotebooks with at least one structural/hygiene flag: {len(issues)} / {len(audit_df)}")


                             notebook  code_cells  md_cells  abs_paths  conv_hits  empty_code  unpaired_code  ends_with_md
       day016_shot_analysis_eda.ipynb           7         2          0          0           0              6          True
      day017_xg_features_part_1.ipynb          10        11          0          0           0              0          True
 day018_xg_geometrical_features.ipynb           6         9          0          0           0              0          True
        day019_passing_networks.ipynb           5         7          0          0           0              0          True
       day020_nfl_offensive_eda.ipynb           4         8          0          0           0              0          True
     day021_nfl_player_features.ipynb           5        10          0          0           0              0          True
        day022_rolling_features.ipynb           4         9          0          0           0              0          True
 day023_per_90_a

### Key Takeaways: Schema Standardization & Hygiene

- **Zero reproducibility or tone violations:** Across all Phase 2 notebooks (`day016` to `day034`), there are **0 hardcoded local paths**, **0 conversational traces**, and **0 empty placeholder cells**.
- **Uniform notebook anatomy:** After adding conceptual Markdown headers to `day016`, `day023`, `day024`, `day029`, `day030` and a closing summary to `day026`, **19 / 19** audited notebooks pass every structural check.

## 3. End-to-End Execution & Automated Test Suite (`pytest`)

**Objective:** Verify that our modular Python codebase (`src/`), automated test suite (`tests/`), and feature pipelines execute cleanly from top to bottom without import or runtime errors.

**Engineering Rationale:** Before tagging `v0.2-eda-features`, every module must import cleanly and pass `pytest` smoke/loader checks.

In [ ]:
import subprocess
import sys

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# 1. Verify modular imports across src/
import src.features.xg_features
import src.features.xg_features_geom
import src.features.nfl_player_features
import src.features.nfl_situational
import src.features.temporal
import src.features.draft_features
import src.models.player_similarity
print("[PASS] All Phase 2 feature & model modules imported cleanly.")

# 2. Run offline smoke tests from tests/test_smoke.py
from tests.test_smoke import test_src_package_imports, test_version
test_src_package_imports()
test_version()
print("[PASS] tests/test_smoke.py: test_src_package_imports & test_version passed.")

# 3. Execute Days 31-34 notebooks end-to-end (non-interactive Agg backend)
import matplotlib
matplotlib.use("Agg")
for nb_name in [
    "day031_draft_features.ipynb",
    "day032_possession_analysis.ipynb",
    "day033_formation_analysis.ipynb",
    "day034_journal_post_phase_2.ipynb",
]:
    nb_obj = json.loads((nb_dir / nb_name).read_text(encoding="utf-8"))
    code_str = "\n".join("".join(c["source"]) for c in nb_obj["cells"] if c["cell_type"] == "code")
    exec(compile(code_str, nb_name, "exec"), {"__name__": "__main__"})
    print(f"[PASS] Executed {nb_name} cleanly.")


### Key Takeaways: End-to-End Execution & Test Audit

- **Core Phase 2 modules & notebooks pass:** All `src/features/` and `src/models/` modules import cleanly, `tests/test_smoke.py` passes, and Days 31 through 34 execute end-to-end without errors.
- **Logged technical debt (`tests/test_loaders.py`):** The Phase 1 live-API integration test references `load_statsbomb` (renamed in `src/data/loaders.py`) and requires network access; flagged for cleanup alongside CI test separation (offline unit vs. online integration tests).

## 4. Phase 2 Technical Retrospective & Journal Summary

**Executive Summary (Days 16-35):**
Phase 2 transformed raw football (StatsBomb) and NFL (`nflverse`) event streams into leakage-free, domain-grounded feature stores and portfolio-ready analytical products.

### Key Technical Achievements
- **Football Feature Engineering:** Built shot geometry (`xg_features.py`, `xg_features_geom.py`), directed passing network centrality (`day019`), PPDA & high-turnover pressing metrics (`day024`), possession vs. Field Tilt separation (`day032`), and active-formation per-90 profiling (`day033`).
- **NFL Feature Engineering:** Engineered rolling EPA/success windows (`temporal.py`), situational leverage splits (`nfl_situational.py`), position-normalized Combine Z-scores & Barnwell Speed Score (`draft_features.py`), and cohort-adjusted career value (`target_wav_per_yr` capped at 9-year saturation).
- **First End-to-End Product & Publication:** Shipped the **FootballScout** similarity engine (`src/models/player_similarity.py`, `docs/footballscout.html`) and published the three-act World Cup 2022 Technical Journal (`docs/phase2-journal.html`).

### Lessons Learned & Phase 3 Readiness
- **Temporal & Censoring Hygiene:** Raw career totals (`w_av`) and static conference labels introduce look-ahead and right-censoring bias; expanding priors and cohort-normalized denominators resolve both cleanly.
- **MNAR Signals:** Combine drill opt-outs and zero-second possession blips carry real signal when flagged rather than blindly imputed.
- **Next Phase (Days 36-50 Modeling):** Feature stores (`xg_features_sb_43_106.parquet`, `draft_features.parquet`, `features_team_temporal_2023.parquet`) are ready for temporal cross-validation, calibrated gradient boosting, and SHAP interpretability.

## 5. Git Hygiene, `.gitignore` Audit & Release `v0.2-eda-features`

**Objective:** Verify that `.gitignore` blocks raw/processed data caches and temporary artifacts, inspect `git status`, and prepare the clean `v0.2-eda-features` release tag.

**Engineering Rationale:** Keeping binary caches out of Git history ensures fast clones and clean diffs across Phase 3.

In [ ]:
git_status = subprocess.run(["git", "status", "--short"], cwd=REPO_ROOT, capture_output=True, text=True)
print("--- Git Working Tree Status ---")
print(git_status.stdout.strip() or "Working tree clean.")


### Key Takeaways: Release `v0.2-eda-features` Ready

- **`.gitignore` verified:** `data/raw/**/*.parquet`, `data/processed/**/*.parquet`, and `reports/figures/*.png` remain excluded from version control while `docs/images/` web assets, `src/` modules, and standardized notebooks are tracked cleanly.
- **Milestone sealed:** Phase 2 (Days 16-35) is complete and tagged as `v0.2-eda-features`.